# QREV v4.0.0 — Corrected cohort extraction and empirical validation

This notebook advances the accepted QREV analytical preflight into the corrected 519-recording cohort stage. It follows the common QGAIN/QADD evaluation architecture: G1–G10, the ten-domain checklist, Panels A–J with explicit N/A handling, support-aware non-imputed ML export, and immutable-freeze governance.

**Scientific contract.** Natural post-speech boundaries come only from `primary_speech / primary`; `strict_speech / primary` supplies SRMR speech support only. The signal is the deterministic 16-kHz mono analysis waveform with global DC removal and no amplitude normalization, denoising, or dereverberation. Boundary measurements are conditional residual-tail descriptors; SRMR is a pinned reverberation-sensitive comparator. None is RT60, EDT, DRR, RIR recovery, room identity, or confirmed echo.

This notebook cannot freeze QREV. It produces empirical evidence for G6–G8 and leaves feature-specific G10 decisions pending independent scientific review.

In [ ]:
from __future__ import annotations

from dataclasses import replace
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import os
import shutil
import subprocess
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import signal, stats
from IPython.display import display, Markdown


def find_project_root() -> Path:
    override = os.environ.get("PAPER1_PROJECT_ROOT", "").strip()
    if override:
        candidate = Path(override).expanduser().resolve()
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
        raise FileNotFoundError(f"PAPER1_PROJECT_ROOT is invalid: {candidate}")
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Open this notebook from inside the Paper 1 repository, "
        "or set PAPER1_PROJECT_ROOT."
    )


ROOT = find_project_root()
REVIEWED_SRC = ROOT / "src"
ORIGINAL_SRC = ROOT / "src"
for path in [REVIEWED_SRC, ORIGINAL_SRC]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from paper1_qc.media import decode_audio_views
from paper1_qc_reviewed.qrev_v400 import (
    ANALYSIS_FEATURES,
    CONDITIONAL_BOUNDARY_FEATURES,
    DEFAULT_PARAMETERS,
    MEASUREMENT_VERSION,
    SRMR_GAMMATONE_VERSION,
    SRMR_UPSTREAM_COMMIT,
    SRMR_VARIANT,
    boundary_envelope_trace,
    extract_qrev,
    feature_registry_frame,
)
from paper1_qc_reviewed.qrev_v400_cohort import (
    analysis_waveform_from_audio_views,
    BOUNDARY_FEATURE_SPECS,
    CANONICAL_PRIMARY_VIEW,
    CANONICAL_PROFILE,
    CANONICAL_STRICT_VIEW,
    COHORT_ORCHESTRATION_VERSION,
    SUPPORT_POLICIES,
    as_bool,
    bootstrap_median_precision,
    canonical_interval_contract,
    deterministic_stratified_sample,
    empirical_feature_summary,
    hash_inventory,
    intervals_for_recording,
    json_safe,
    model_interface_frame,
    participant_balanced_resampling,
    participant_balanced_summary,
    policy_values,
    repeated_recording_persistence,
    resolve_media_path,
    sha256_file,
    shift_primary_offsets,
    summarize_delete_one,
    delete_one_boundary_grid,
    support_policy_availability,
    write_json,
)

# ---------------------------- execution controls ----------------------------
RUN_PACKAGE_TESTS = True
RUN_COHORT_EXTRACTION = False  # installer changes this to True in the local run copy
VERIFY_MEDIA_HASHES = True
REBUILD_CHECKPOINTS = False
RUN_COHORT_ROBUSTNESS = True
RUN_SRMR_BANDWIDTH_CHARACTERIZATION = True
BUILD_GALLERY = True

MAX_ROBUSTNESS_RECORDINGS = 96
BOOTSTRAP_ITERATIONS = 400
PARTICIPANT_BALANCED_ITERATIONS = 1000
SRMR_BANDWIDTH_RECORDINGS = 24
GALLERY_RECORDING_LIMIT = 10

MEDIA_ROOT_OVERRIDE = None  # installer inserts the local Bamboo_passage_only path
MEDIA_PATH_MAP = {}

PUBLISH_AND_FREEZE = False
SCIENTIFIC_REVIEW_DECISION = "PENDING"
SCIENTIFIC_REVIEWER = ""
SCIENTIFIC_REVIEW_RATIONALE = ""

PARAMETERS = DEFAULT_PARAMETERS
FS = PARAMETERS.analysis_sample_rate_hz

LEGACY_MAIN = ROOT / "MAIN outputs"
STAGE = ROOT / "outputs/reviewed" / "reverberation" / MEASUREMENT_VERSION
TABLES = STAGE / "tables"
LEDGERS = STAGE / "ledgers"
VALIDATION = STAGE / "validation"
FIGURES = STAGE / "figures"
GALLERIES = STAGE / "galleries"
AUDIT = STAGE / "audit"
MANIFESTS = STAGE / "manifests"
CHECKPOINTS = STAGE / "checkpoints"
for directory in [
    TABLES, LEDGERS, VALIDATION, FIGURES, GALLERIES,
    AUDIT, MANIFESTS, CHECKPOINTS,
]:
    directory.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("default")
pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 240)

print("Project root:", ROOT)
print("Measurement:", MEASUREMENT_VERSION)
print("Orchestration:", COHORT_ORCHESTRATION_VERSION)
print("Run cohort:", RUN_COHORT_EXTRACTION)
print("Stage:", STAGE)

In [ ]:
def save_table(frame: pd.DataFrame, path_without_suffix: Path, *, parquet: bool = True) -> dict:
    path_without_suffix = Path(path_without_suffix)
    path_without_suffix.parent.mkdir(parents=True, exist_ok=True)
    csv_path = path_without_suffix.with_suffix(".csv")
    frame.to_csv(csv_path, index=False)
    result = {"csv": str(csv_path), "csv_sha256": sha256_file(csv_path)}
    if parquet:
        parquet_path = path_without_suffix.with_suffix(".parquet")
        try:
            frame.to_parquet(parquet_path, index=False)
            result.update({"parquet": str(parquet_path), "parquet_sha256": sha256_file(parquet_path)})
        except Exception as exc:
            result["parquet_error"] = f"{type(exc).__name__}: {exc}"
    return result


def save_figure_bundle(figure, *, stem: str, panel: str, source_data: pd.DataFrame, caption: str, scientific_question: str, provenance: dict | None = None) -> dict:
    png = FIGURES / f"{stem}.png"
    svg = FIGURES / f"{stem}.svg"
    pdf = FIGURES / f"{stem}.pdf"
    source = FIGURES / f"{stem}.source.csv"
    caption_path = FIGURES / f"{stem}.caption.md"
    provenance_path = FIGURES / f"{stem}.provenance.json"
    figure.tight_layout()
    figure.savefig(png, dpi=300, bbox_inches="tight")
    figure.savefig(svg, bbox_inches="tight")
    figure.savefig(pdf, bbox_inches="tight")
    source_data.to_csv(source, index=False)
    caption_path.write_text(caption.strip() + "\n", encoding="utf-8")
    write_json(
        {
            "panel": panel,
            "figure_id": stem,
            "scientific_question": scientific_question,
            "measurement_version": MEASUREMENT_VERSION,
            "implementation_sha256": sha256_file(REVIEWED_SRC / "paper1_qc_reviewed" / "qrev_v400.py"),
            "orchestration_sha256": sha256_file(REVIEWED_SRC / "paper1_qc_reviewed" / "qrev_v400_cohort.py"),
            "source_csv": source.name,
            "source_csv_sha256": sha256_file(source),
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "feature_values_recomputed_by_figure": False,
            **(provenance or {}),
        },
        provenance_path,
    )
    return {
        "panel": panel,
        "figure_id": stem,
        "png": png.name,
        "svg": svg.name,
        "pdf": pdf.name,
        "source": source.name,
        "caption": caption_path.name,
        "provenance": provenance_path.name,
    }


def subject_column_for(frame: pd.DataFrame) -> str:
    for column in ["SubjectID", "subject_id", "subject", "ID_norm"]:
        if column in frame and frame[column].notna().any():
            return column
    raise ValueError("No subject identifier column was found")


def date_column_for(frame: pd.DataFrame) -> str:
    for column in ["recording_date_analysis", "Recording date", "date_parsed", "recording_date"]:
        if column in frame and frame[column].notna().any():
            return column
    raise ValueError("No recording-date column was found")


def checkpoint_paths(recording_id: str) -> dict[str, Path]:
    safe = str(recording_id).replace("/", "_").replace("\\", "_")
    return {
        "record": CHECKPOINTS / "records" / f"{safe}.json",
        "boundary": CHECKPOINTS / "boundary_parts" / f"{safe}.parquet",
        "media": CHECKPOINTS / "media_audit" / f"{safe}.json",
        "error": CHECKPOINTS / "errors" / f"{safe}.json",
    }


def checkpoint_complete(paths: dict[str, Path]) -> bool:
    return all(paths[key].exists() for key in ["record", "boundary", "media"])


def media_hash_column(frame: pd.DataFrame) -> str | None:
    for column in ["media_sha256", "selected_media_sha256", "sha256"]:
        if column in frame:
            return column
    return None


def finite_abs_delta(a, b):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    mask = np.isfinite(a) & np.isfinite(b)
    return np.abs(a[mask] - b[mask])


def comparison_summary(long_frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if long_frame.empty:
        return pd.DataFrame()
    for keys, local in long_frame.groupby(["variant", "feature"], sort=True):
        paired = local.loc[local["baseline_available"] & local["variant_available"]].copy()
        delta = pd.to_numeric(paired["absolute_delta"], errors="coerce").dropna()
        rows.append({
            "variant": keys[0],
            "feature": keys[1],
            "recording_count": local["logical_recording_id"].nunique(),
            "baseline_available_n": int(local["baseline_available"].sum()),
            "variant_available_n": int(local["variant_available"].sum()),
            "availability_agreement_fraction": float((local["baseline_available"] == local["variant_available"]).mean()),
            "paired_finite_n": len(delta),
            "median_absolute_delta": float(delta.median()) if len(delta) else np.nan,
            "p95_absolute_delta": float(delta.quantile(0.95)) if len(delta) else np.nan,
            "maximum_absolute_delta": float(delta.max()) if len(delta) else np.nan,
        })
    return pd.DataFrame(rows)


def lowpass_waveform(waveform: np.ndarray, cutoff_hz: float, fs: int) -> np.ndarray:
    if cutoff_hz >= fs / 2 - 1:
        return np.asarray(waveform, dtype=np.float64).copy()
    sos = signal.butter(8, float(cutoff_hz), btype="lowpass", fs=int(fs), output="sos")
    return signal.sosfiltfilt(sos, np.asarray(waveform, dtype=np.float64))


def select_evenly(frame: pd.DataFrame, count: int, sort_columns: list[str]) -> pd.DataFrame:
    if len(frame) <= count:
        return frame.copy()
    ordered = frame.sort_values(sort_columns).reset_index(drop=True)
    positions = np.linspace(0, len(ordered) - 1, count).round().astype(int)
    return ordered.iloc[np.unique(positions)].copy()


figure_index_rows = []

In [ ]:
# Verify accepted hotfix preflight and complete A-C artifact bundles.
preflight_manifest_path = MANIFESTS / "qrev_v400_preflight_manifest.json"
if not preflight_manifest_path.exists():
    raise FileNotFoundError(
        "The accepted QREV v4.0.0 hotfix preflight manifest is missing. "
        "Run and save the corrected preflight notebook first."
    )
preflight_manifest = json.loads(preflight_manifest_path.read_text(encoding="utf-8"))
preflight_checks = pd.read_csv(TABLES / "qrev_v400_preflight_all_checks.csv")
preflight_gates = pd.read_csv(TABLES / "qrev_v400_gate_summary.csv")
preflight_figure_index = pd.read_csv(TABLES / "qrev_v400_preflight_figure_index.csv")

preflight_bundle_rows = []
for stem in ["A_construct_response", "B_discriminant_specificity", "C_transformation_contract"]:
    expected = [
        FIGURES / f"{stem}.png",
        FIGURES / f"{stem}.svg",
        FIGURES / f"{stem}.pdf",
        FIGURES / f"{stem}.source.csv",
        FIGURES / f"{stem}.caption.md",
        FIGURES / f"{stem}.provenance.json",
    ]
    preflight_bundle_rows.append({
        "figure_id": stem,
        "all_artifacts_exist": all(path.exists() for path in expected),
        "missing": ";".join(path.name for path in expected if not path.exists()),
    })
preflight_bundle_check = pd.DataFrame(preflight_bundle_rows)

preflight_ok = bool(
    preflight_manifest.get("preflight_blocking_checks_pass", False)
    and preflight_manifest.get("srmr_runtime_available", False)
    and preflight_manifest.get("global_dc_removal_enforced", False)
    and preflight_manifest.get("preflight_hotfix_revision") == "dc-contract-hotfix-1"
    and not preflight_checks["passed"].map(lambda x: str(x).lower() in {"false", "0"}).any()
    and preflight_bundle_check["all_artifacts_exist"].all()
)
if not preflight_ok:
    display(preflight_manifest)
    display(preflight_checks.loc[~preflight_checks["passed"].astype(bool)])
    display(preflight_bundle_check)
    raise RuntimeError("Accepted QREV hotfix preflight evidence is incomplete")

for row in preflight_figure_index.to_dict("records"):
    figure_index_rows.append({**row, "stage": "PREFLIGHT", "status": "COMPLETE"})

package_tests_passed = True
package_test_output = "not_requested"
if RUN_PACKAGE_TESTS:
    command = [
        sys.executable, "-m", "pytest",
        "tests/test_qrev_v400.py",
        "tests/test_qrev_v400_cohort.py",
        "-q", "--disable-warnings",
    ]
    test_env = os.environ.copy()
    test_env["PYTHONPATH"] = os.pathsep.join([
        str(REVIEWED_SRC), str(ORIGINAL_SRC), test_env.get("PYTHONPATH", "")
    ])
    completed = subprocess.run(command, cwd=ROOT, capture_output=True, text=True, env=test_env)
    package_test_output = completed.stdout + "\n" + completed.stderr
    (AUDIT / "qrev_v400_cohort_package_test_output.txt").write_text(package_test_output, encoding="utf-8")
    package_tests_passed = completed.returncode == 0
    if not package_tests_passed:
        print(package_test_output)
        raise RuntimeError("Reviewed QREV cohort package tests failed")

print("Accepted preflight verified.")
print(package_test_output[-1500:])
display(preflight_gates)
display(preflight_bundle_check)

In [ ]:
# Load frozen cohort inputs and enforce exact primary/strict interval contract.
DATA_FREEZE = LEGACY_MAIN / "00_DATA_FREEZE" / "v1"
SEGMENTATION_FREEZE = LEGACY_MAIN / "01_SEGMENTATION_FREEZE" / "v1"
recordings_path = DATA_FREEZE / "frozen_bamboo_recordings.csv"
decisions_path = SEGMENTATION_FREEZE / "frozen_segmentation_decisions.csv"
intervals_path = SEGMENTATION_FREEZE / "frozen_segmentation_intervals.csv"
for path in [recordings_path, decisions_path, intervals_path]:
    if not path.exists():
        raise FileNotFoundError(path)

frozen_recordings = pd.read_csv(recordings_path, low_memory=False)
frozen_decisions = pd.read_csv(decisions_path, low_memory=False)
frozen_intervals = pd.read_csv(intervals_path, low_memory=False)

recording_eligible = as_bool(frozen_recordings["freeze_included"])
segmentation_eligible = as_bool(frozen_decisions["segmentation_analysis_eligible"])
frozen = frozen_recordings.loc[recording_eligible].merge(
    frozen_decisions.loc[segmentation_eligible, [
        "logical_recording_id", "segmentation_analysis_eligible", "segmentation_decision_source"
    ]],
    on="logical_recording_id",
    how="inner",
    validate="one_to_one",
)
frozen["logical_recording_id"] = frozen["logical_recording_id"].astype(str)
frozen = frozen.sort_values("logical_recording_id").reset_index(drop=True)

canonical_tables, interval_contract, primary_strict_pairing = canonical_interval_contract(
    frozen_decisions, frozen_intervals
)
primary_table = canonical_tables[CANONICAL_PRIMARY_VIEW]
strict_table = canonical_tables[CANONICAL_STRICT_VIEW]

input_artifacts = pd.DataFrame([
    {"artifact": "frozen_bamboo_recordings.csv", "path": str(recordings_path), "sha256": sha256_file(recordings_path)},
    {"artifact": "frozen_segmentation_decisions.csv", "path": str(decisions_path), "sha256": sha256_file(decisions_path)},
    {"artifact": "frozen_segmentation_intervals.csv", "path": str(intervals_path), "sha256": sha256_file(intervals_path)},
    {"artifact": "qrev_v400.py", "path": str(REVIEWED_SRC / "paper1_qc_reviewed" / "qrev_v400.py"), "sha256": sha256_file(REVIEWED_SRC / "paper1_qc_reviewed" / "qrev_v400.py")},
    {"artifact": "qrev_v400_cohort.py", "path": str(REVIEWED_SRC / "paper1_qc_reviewed" / "qrev_v400_cohort.py"), "sha256": sha256_file(REVIEWED_SRC / "paper1_qc_reviewed" / "qrev_v400_cohort.py")},
])

save_table(input_artifacts, AUDIT / "qrev_v400_input_artifacts", parquet=False)
save_table(interval_contract, AUDIT / "qrev_v400_canonical_interval_contract", parquet=False)
save_table(primary_strict_pairing, AUDIT / "qrev_v400_primary_strict_pairing")
save_table(primary_table, LEDGERS / "qrev_v400_canonical_primary_speech_intervals")
save_table(strict_table, LEDGERS / "qrev_v400_canonical_strict_speech_intervals")
save_table(feature_registry_frame(), TABLES / "qrev_v400_feature_registry", parquet=False)

subject_column = subject_column_for(frozen)
date_column = date_column_for(frozen)
media_sha_column = media_hash_column(frozen)

input_checks = pd.DataFrame([
    {"gate": "G1", "check": "corrected cohort count", "passed": len(frozen) == 519, "observed": len(frozen), "required": 519},
    {"gate": "G1", "check": "participant count", "passed": frozen[subject_column].nunique(dropna=True) == 224, "observed": frozen[subject_column].nunique(dropna=True), "required": 224},
    {"gate": "G1", "check": "canonical interval contract", "passed": interval_contract["contract_pass"].all(), "observed": interval_contract["contract_pass"].tolist(), "required": "all true"},
    {"gate": "G1", "check": "all eligible recordings have primary intervals", "passed": primary_table["logical_recording_id"].nunique() == len(frozen), "observed": primary_table["logical_recording_id"].nunique(), "required": len(frozen)},
    {"gate": "G1", "check": "all eligible recordings have strict intervals", "passed": strict_table["logical_recording_id"].nunique() == len(frozen), "observed": strict_table["logical_recording_id"].nunique(), "required": len(frozen)},
    {"gate": "G1", "check": "strict intervals paired inside primary intervals", "passed": primary_strict_pairing["pair_complete"].all() and primary_strict_pairing["strict_inside_primary"].all(), "observed": int(primary_strict_pairing["strict_inside_primary"].sum()), "required": len(primary_strict_pairing)},
])
if not input_checks["passed"].all():
    display(input_checks)
    raise RuntimeError("Frozen QREV cohort contract failed")
save_table(input_checks, VALIDATION / "qrev_v400_g1_input_checks", parquet=False)

display(input_checks)
display(interval_contract)

In [ ]:
# Corrected recording-level extraction with restart-safe checkpoints.
ffmpeg = shutil.which("ffmpeg")
ffprobe = shutil.which("ffprobe")
if RUN_COHORT_EXTRACTION and (not ffmpeg or not ffprobe):
    raise RuntimeError("ffmpeg and ffprobe must be available on PATH")

if REBUILD_CHECKPOINTS and CHECKPOINTS.exists():
    shutil.rmtree(CHECKPOINTS)
for directory in [CHECKPOINTS / "records", CHECKPOINTS / "boundary_parts", CHECKPOINTS / "media_audit", CHECKPOINTS / "errors"]:
    directory.mkdir(parents=True, exist_ok=True)

record_rows = []
boundary_parts = []
media_audit_rows = []
error_rows = []

if RUN_COHORT_EXTRACTION:
    started = time.time()
    for position, frozen_row in enumerate(frozen.itertuples(index=False), start=1):
        recording_id = str(frozen_row.logical_recording_id)
        paths = checkpoint_paths(recording_id)
        try:
            if checkpoint_complete(paths) and not REBUILD_CHECKPOINTS:
                record_rows.append(json.loads(paths["record"].read_text(encoding="utf-8")))
                boundary_parts.append(pd.read_parquet(paths["boundary"]))
                media_audit_rows.append(json.loads(paths["media"].read_text(encoding="utf-8")))
                continue

            raw_media_path = getattr(frozen_row, "media_path")
            media_path = resolve_media_path(
                raw_media_path,
                media_root_override=Path(MEDIA_ROOT_OVERRIDE) if MEDIA_ROOT_OVERRIDE else None,
                media_path_map=MEDIA_PATH_MAP,
            )
            observed_sha = sha256_file(media_path) if VERIFY_MEDIA_HASHES else "not_checked"
            expected_sha = str(getattr(frozen_row, media_sha_column)) if media_sha_column else ""
            hash_match = bool(not VERIFY_MEDIA_HASHES or not expected_sha or observed_sha == expected_sha)
            if not hash_match:
                raise RuntimeError(
                    f"Media SHA-256 mismatch for {recording_id}: expected {expected_sha}, observed {observed_sha}"
                )

            views = decode_audio_views(
                media_path,
                ffmpeg=ffmpeg,
                ffprobe=ffprobe,
                analysis_rate=FS,
            )
            waveform = analysis_waveform_from_audio_views(views)
            primary, _ = intervals_for_recording(primary_table, recording_id)
            strict, _ = intervals_for_recording(strict_table, recording_id)
            result = extract_qrev(
                waveform,
                FS,
                primary_speech=primary,
                strict_speech=strict,
                logical_recording_id=recording_id,
                parameters=PARAMETERS,
                compute_srmr=True,
            )
            record = dict(result.recording)
            for column in [
                "SubjectID", "recording_date_analysis", "Recording date", "diagnosis_analysis",
                "selected_media_file_name", "selected_media_extension", "media_path", "media_sha256",
            ]:
                if hasattr(frozen_row, column):
                    record[column] = getattr(frozen_row, column)
            record["qrev_media_path_resolved"] = str(media_path)
            record["qrev_media_sha256_observed"] = observed_sha
            record["qrev_media_sha256_match"] = hash_match
            record["qrev_analysis_duration_sec"] = len(waveform) / FS
            record["qrev_native_sample_rate_hz"] = int(views.sample_rate_native)

            ledger = result.boundary_ledger.copy()
            if len(ledger):
                ledger["media_sha256"] = observed_sha
                ledger["measurement_version"] = MEASUREMENT_VERSION

            media_audit = {
                "logical_recording_id": recording_id,
                "media_path": str(media_path),
                "expected_sha256": expected_sha,
                "observed_sha256": observed_sha,
                "sha256_match": hash_match,
                "analysis_sample_rate_hz": FS,
                "analysis_duration_sec": len(waveform) / FS,
            }

            paths["record"].write_text(json.dumps(json_safe(record), indent=2), encoding="utf-8")
            ledger.to_parquet(paths["boundary"], index=False)
            paths["media"].write_text(json.dumps(json_safe(media_audit), indent=2), encoding="utf-8")
            record_rows.append(record)
            boundary_parts.append(ledger)
            media_audit_rows.append(media_audit)

            if position == 1 or position % 20 == 0 or position == len(frozen):
                elapsed = time.time() - started
                rate = elapsed / position
                remaining_min = rate * (len(frozen) - position) / 60.0
                print(f"[{position:03d}/{len(frozen)}] {recording_id} | ETA {remaining_min:.1f} min")
        except Exception as exc:
            error = {
                "logical_recording_id": recording_id,
                "error_type": type(exc).__name__,
                "message": str(exc),
            }
            error_rows.append(error)
            paths["error"].write_text(json.dumps(error, indent=2), encoding="utf-8")
            print("ERROR", recording_id, type(exc).__name__, exc)

recording_table = pd.DataFrame(record_rows)
boundary_ledger = pd.concat(boundary_parts, ignore_index=True) if boundary_parts else pd.DataFrame()
media_audit = pd.DataFrame(media_audit_rows)
extraction_errors = pd.DataFrame(error_rows, columns=["logical_recording_id", "error_type", "message"])

if RUN_COHORT_EXTRACTION:
    if recording_table["logical_recording_id"].duplicated().any():
        raise RuntimeError("Recording-level QREV rows are duplicated")
    recording_table = frozen.merge(recording_table, on="logical_recording_id", how="left", suffixes=("", "__qrev"), validate="one_to_one")

save_table(recording_table, TABLES / "qrev_v400_analysis_features")
save_table(boundary_ledger, LEDGERS / "qrev_v400_boundary_ledger")
save_table(media_audit, AUDIT / "qrev_v400_media_hash_audit", parquet=False)
save_table(extraction_errors, AUDIT / "qrev_v400_extraction_errors", parquet=False)

extraction_checks = pd.DataFrame([
    {"gate": "G7", "check": "all frozen recordings extracted", "passed": len(recording_table) == len(frozen), "observed": len(recording_table), "required": len(frozen)},
    {"gate": "G7", "check": "no unexplained extraction errors", "passed": extraction_errors.empty, "observed": len(extraction_errors), "required": 0},
    {"gate": "G7", "check": "all media hashes verified", "passed": bool(len(media_audit) == len(frozen) and media_audit.get("sha256_match", pd.Series(dtype=bool)).astype(bool).all()), "observed": int(media_audit.get("sha256_match", pd.Series(dtype=bool)).astype(bool).sum()), "required": len(frozen)},
    {"gate": "G2", "check": "global DC removal applied", "passed": bool(len(recording_table) and recording_table.get("qrev_global_dc_removal_applied", pd.Series(dtype=bool)).astype(bool).all()), "observed": int(recording_table.get("qrev_global_dc_removal_applied", pd.Series(dtype=bool)).astype(bool).sum()), "required": len(frozen)},
])
if RUN_COHORT_EXTRACTION and not extraction_checks["passed"].all():
    display(extraction_checks)
save_table(extraction_checks, VALIDATION / "qrev_v400_extraction_checks", parquet=False)
display(extraction_checks)
display(recording_table.head())

In [ ]:
# G2/G6 — exact reconstruction, support-policy comparison, deletion and bootstrap precision.
reconstruction_rows = []
if RUN_COHORT_EXTRACTION and len(recording_table):
    ledger_groups = {str(key): group for key, group in boundary_ledger.groupby("logical_recording_id", sort=False)}
    for row in recording_table.itertuples(index=False):
        recording_id = str(row.logical_recording_id)
        local = ledger_groups.get(recording_id, pd.DataFrame())
        for feature, spec in BOUNDARY_FEATURE_SPECS.items():
            if len(local):
                eligible = as_bool(local[spec["flag"]])
                values = pd.to_numeric(local.loc[eligible, spec["ledger_value"]], errors="coerce").dropna()
            else:
                values = pd.Series(dtype=float)
            reconstructed = float(values.median()) if len(values) else np.nan
            stored = pd.to_numeric(pd.Series([getattr(row, spec["recording_raw"])]), errors="coerce").iloc[0]
            both_missing = pd.isna(reconstructed) and pd.isna(stored)
            exact = bool(both_missing or (np.isfinite(reconstructed) and np.isclose(reconstructed, stored, atol=1e-12, rtol=0.0)))
            reconstruction_rows.append({
                "logical_recording_id": recording_id,
                "feature": feature,
                "eligible_boundary_count": len(values),
                "reconstructed_raw_estimate": reconstructed,
                "stored_raw_estimate": stored,
                "exact_match": exact,
            })
reconstruction_audit = pd.DataFrame(reconstruction_rows)
save_table(reconstruction_audit, AUDIT / "qrev_v400_reconstruction_audit")

support_policy_summary = support_policy_availability(recording_table) if len(recording_table) else pd.DataFrame()
save_table(support_policy_summary, VALIDATION / "qrev_v400_support_policy_availability")
if len(recording_table):
    for policy in SUPPORT_POLICIES:
        save_table(policy_values(recording_table, minimum_boundary_count=policy), TABLES / f"qrev_v400_policy_min_{policy}_features")

delete_one_grid = delete_one_boundary_grid(boundary_ledger) if RUN_COHORT_ROBUSTNESS else pd.DataFrame()
delete_one_summary = summarize_delete_one(delete_one_grid)
save_table(delete_one_grid, VALIDATION / "qrev_v400_delete_one_boundary_grid")
save_table(delete_one_summary, VALIDATION / "qrev_v400_delete_one_boundary_summary")

bootstrap_precision = bootstrap_median_precision(
    boundary_ledger,
    iterations=BOOTSTRAP_ITERATIONS,
) if RUN_COHORT_ROBUSTNESS else pd.DataFrame()
save_table(bootstrap_precision, VALIDATION / "qrev_v400_bootstrap_median_precision")

reconstruction_pass = bool(
    len(reconstruction_audit)
    and reconstruction_audit["exact_match"].all()
    and reconstruction_audit["logical_recording_id"].nunique() == len(recording_table)
)
g6_foundational_checks = pd.DataFrame([
    {"gate": "G2", "check": "recording estimates reconstruct from boundary ledger", "passed": reconstruction_pass, "observed": int(reconstruction_audit.get("exact_match", pd.Series(dtype=bool)).sum()), "required": len(recording_table) * 3},
    {"gate": "G6", "check": "2/3/4 boundary policies quantified", "passed": set(support_policy_summary.get("minimum_boundary_count", [])) >= {2, 3, 4}, "observed": sorted(set(support_policy_summary.get("minimum_boundary_count", []))), "required": [2, 3, 4]},
    {"gate": "G6", "check": "delete-one-boundary evidence generated", "passed": len(delete_one_summary) > 0, "observed": len(delete_one_summary), "required": ">0"},
    {"gate": "G6", "check": "bootstrap precision evidence generated", "passed": len(bootstrap_precision) > 0, "observed": len(bootstrap_precision), "required": ">0"},
])
save_table(g6_foundational_checks, VALIDATION / "qrev_v400_g2_g6_foundational_checks", parquet=False)
display(g6_foundational_checks)
display(support_policy_summary)
display(delete_one_summary)

In [ ]:
# G6 — boundary, floor, horizon, threshold, frame, early-window and decay-window sensitivity.
robustness_long_rows = []
robustness_errors = []

if RUN_COHORT_EXTRACTION and RUN_COHORT_ROBUSTNESS and len(recording_table):
    support_frame = recording_table[[
        "logical_recording_id", "qrev_tail_valid_boundary_count",
        "qrev_persistence_recording_median_censored", "qrev_family_status",
    ]].copy()
    support_frame["support_band"] = pd.cut(
        pd.to_numeric(support_frame["qrev_tail_valid_boundary_count"], errors="coerce").fillna(0),
        bins=[-0.1, 1.9, 3.9, 5.9, np.inf],
        labels=["lt2", "2to3", "4to5", "ge6"],
    ).astype(str)
    robustness_sample = deterministic_stratified_sample(
        support_frame,
        maximum_rows=MAX_ROBUSTNESS_RECORDINGS,
        stratum_columns=["support_band", "qrev_persistence_recording_median_censored", "qrev_family_status"],
    )
    save_table(robustness_sample, VALIDATION / "qrev_v400_robustness_sample", parquet=False)

    parameter_variants = [
        ("baseline", PARAMETERS, 0.0),
        ("offset_minus_100ms", PARAMETERS, -100.0),
        ("offset_minus_50ms", PARAMETERS, -50.0),
        ("offset_plus_50ms", PARAMETERS, 50.0),
        ("offset_plus_100ms", PARAMETERS, 100.0),
        ("floor_600_900ms", replace(PARAMETERS, floor_start_ms=600.0, floor_end_ms=900.0), 0.0),
        ("floor_800_1100ms", replace(PARAMETERS, floor_start_ms=800.0, floor_end_ms=1100.0), 0.0),
        ("horizon_400ms", replace(PARAMETERS, persistence_horizon_ms=400.0), 0.0),
        ("horizon_500ms", replace(PARAMETERS, persistence_horizon_ms=500.0), 0.0),
        ("threshold_2db", replace(PARAMETERS, persistence_threshold_db=2.0), 0.0),
        ("threshold_4db", replace(PARAMETERS, persistence_threshold_db=4.0), 0.0),
        ("consecutive_2", replace(PARAMETERS, persistence_consecutive_frames=2), 0.0),
        ("consecutive_4", replace(PARAMETERS, persistence_consecutive_frames=4), 0.0),
        ("frame_20ms_hop10ms", replace(PARAMETERS, frame_length_ms=20.0, frame_hop_ms=10.0), 0.0),
        ("frame_40ms_hop10ms", replace(PARAMETERS, frame_length_ms=40.0, frame_hop_ms=10.0), 0.0),
        ("early_80ms", replace(PARAMETERS, early_tail_end_ms=80.0, minimum_early_frame_count=4), 0.0),
        ("early_120ms", replace(PARAMETERS, early_tail_end_ms=120.0, minimum_early_frame_count=7), 0.0),
        ("decay_200ms", replace(PARAMETERS, decay_end_ms=200.0, minimum_decay_frame_count=15), 0.0),
        ("decay_400ms", replace(PARAMETERS, decay_end_ms=400.0, minimum_decay_frame_count=25), 0.0),
    ]

    frozen_lookup = frozen.set_index("logical_recording_id")
    baseline_lookup = recording_table.set_index("logical_recording_id")
    for sample_row in robustness_sample.itertuples(index=False):
        recording_id = str(sample_row.logical_recording_id)
        try:
            source_row = frozen_lookup.loc[recording_id]
            media_path = resolve_media_path(
                source_row["media_path"],
                media_root_override=Path(MEDIA_ROOT_OVERRIDE) if MEDIA_ROOT_OVERRIDE else None,
                media_path_map=MEDIA_PATH_MAP,
            )
            views = decode_audio_views(media_path, ffmpeg=ffmpeg, ffprobe=ffprobe, analysis_rate=FS)
            waveform = analysis_waveform_from_audio_views(views)
            primary, _ = intervals_for_recording(primary_table, recording_id)
            strict, _ = intervals_for_recording(strict_table, recording_id)
            baseline = baseline_lookup.loc[recording_id]

            for variant_name, variant_parameters, offset_shift_ms in parameter_variants:
                variant_primary = shift_primary_offsets(primary, offset_shift_ms) if offset_shift_ms else primary
                extraction = extract_qrev(
                    waveform,
                    FS,
                    primary_speech=variant_primary,
                    strict_speech=strict,
                    logical_recording_id=recording_id,
                    parameters=variant_parameters,
                    compute_srmr=False,
                )
                for feature in CONDITIONAL_BOUNDARY_FEATURES:
                    baseline_value = pd.to_numeric(pd.Series([baseline[feature]]), errors="coerce").iloc[0]
                    variant_value = pd.to_numeric(pd.Series([extraction.recording[feature]]), errors="coerce").iloc[0]
                    robustness_long_rows.append({
                        "logical_recording_id": recording_id,
                        "variant": variant_name,
                        "feature": feature,
                        "baseline_value": baseline_value,
                        "variant_value": variant_value,
                        "baseline_available": bool(np.isfinite(baseline_value)),
                        "variant_available": bool(np.isfinite(variant_value)),
                        "absolute_delta": abs(variant_value - baseline_value) if np.isfinite(baseline_value) and np.isfinite(variant_value) else np.nan,
                        "variant_status": extraction.recording.get(f"{feature}_status", ""),
                    })
        except Exception as exc:
            robustness_errors.append({"logical_recording_id": recording_id, "error_type": type(exc).__name__, "message": str(exc)})

robustness_long = pd.DataFrame(robustness_long_rows)
robustness_summary = comparison_summary(robustness_long)
robustness_error_table = pd.DataFrame(robustness_errors, columns=["logical_recording_id", "error_type", "message"])
save_table(robustness_long, VALIDATION / "qrev_v400_parameter_sensitivity_long")
save_table(robustness_summary, VALIDATION / "qrev_v400_parameter_sensitivity_summary")
save_table(robustness_error_table, AUDIT / "qrev_v400_robustness_errors", parquet=False)

g6_sensitivity_checks = pd.DataFrame([
    {"gate": "G6", "check": "offset sensitivity grid complete", "passed": set(robustness_long.get("variant", [])) >= {"offset_minus_100ms", "offset_minus_50ms", "offset_plus_50ms", "offset_plus_100ms"}, "observed": sorted(set(robustness_long.get("variant", []))), "required": "-100/-50/+50/+100 ms"},
    {"gate": "G6", "check": "independent floor variants complete", "passed": set(robustness_long.get("variant", [])) >= {"floor_600_900ms", "floor_800_1100ms"}, "observed": sorted(v for v in set(robustness_long.get("variant", [])) if str(v).startswith("floor_")), "required": "two nonoverlapping reasonable variants"},
    {"gate": "G6", "check": "persistence sensitivity grid complete", "passed": set(robustness_long.get("variant", [])) >= {"horizon_400ms", "horizon_500ms", "threshold_2db", "threshold_4db", "consecutive_2", "consecutive_4"}, "observed": "generated", "required": "horizon/threshold/consecutive"},
    {"gate": "G6", "check": "frame and estimator-window variants complete", "passed": set(robustness_long.get("variant", [])) >= {"frame_20ms_hop10ms", "frame_40ms_hop10ms", "early_80ms", "early_120ms", "decay_200ms", "decay_400ms"}, "observed": "generated", "required": "frame/early/decay variants"},
    {"gate": "G6", "check": "no robustness execution errors", "passed": robustness_error_table.empty, "observed": len(robustness_error_table), "required": 0},
])
save_table(g6_sensitivity_checks, VALIDATION / "qrev_v400_g6_sensitivity_checks", parquet=False)
display(g6_sensitivity_checks)
display(robustness_summary.head(30))

In [ ]:
# G5 — SRMR bandwidth characterization on a deterministic support-qualified subset.
srmr_bandwidth_rows = []
srmr_bandwidth_errors = []
if RUN_COHORT_EXTRACTION and RUN_SRMR_BANDWIDTH_CHARACTERIZATION and len(recording_table):
    eligible_srmr = recording_table.loc[
        pd.to_numeric(recording_table["qrev_srmr_norm"], errors="coerce").notna(),
        ["logical_recording_id", "qrev_srmr_norm", "qrev_srmr_strict_speech_support_sec"],
    ].copy()
    eligible_srmr["support_band"] = pd.qcut(
        pd.to_numeric(eligible_srmr["qrev_srmr_strict_speech_support_sec"], errors="coerce"),
        q=min(4, max(1, eligible_srmr["qrev_srmr_strict_speech_support_sec"].nunique())),
        duplicates="drop",
    ).astype(str)
    srmr_sample = deterministic_stratified_sample(
        eligible_srmr,
        maximum_rows=SRMR_BANDWIDTH_RECORDINGS,
        stratum_columns=["support_band"],
    )
    save_table(srmr_sample, VALIDATION / "qrev_v400_srmr_bandwidth_sample", parquet=False)
    frozen_lookup = frozen.set_index("logical_recording_id")
    for row in srmr_sample.itertuples(index=False):
        recording_id = str(row.logical_recording_id)
        try:
            source_row = frozen_lookup.loc[recording_id]
            media_path = resolve_media_path(
                source_row["media_path"],
                media_root_override=Path(MEDIA_ROOT_OVERRIDE) if MEDIA_ROOT_OVERRIDE else None,
                media_path_map=MEDIA_PATH_MAP,
            )
            views = decode_audio_views(media_path, ffmpeg=ffmpeg, ffprobe=ffprobe, analysis_rate=FS)
            waveform = analysis_waveform_from_audio_views(views)
            primary, _ = intervals_for_recording(primary_table, recording_id)
            strict, _ = intervals_for_recording(strict_table, recording_id)
            for cutoff in [3000.0, 4000.0, 6000.0, 7999.0]:
                transformed = lowpass_waveform(waveform, cutoff, FS)
                result = extract_qrev(
                    transformed,
                    FS,
                    primary_speech=primary,
                    strict_speech=strict,
                    logical_recording_id=recording_id,
                    parameters=PARAMETERS,
                    compute_srmr=True,
                )
                score = pd.to_numeric(pd.Series([result.recording["qrev_srmr_norm"]]), errors="coerce").iloc[0]
                srmr_bandwidth_rows.append({
                    "logical_recording_id": recording_id,
                    "lowpass_cutoff_hz": cutoff,
                    "qrev_srmr_norm": score,
                    "status": result.recording["qrev_srmr_norm_status"],
                    "baseline_qrev_srmr_norm": float(row.qrev_srmr_norm),
                    "absolute_delta_from_baseline": abs(score - float(row.qrev_srmr_norm)) if np.isfinite(score) else np.nan,
                    "relative_delta_from_baseline": abs(score - float(row.qrev_srmr_norm)) / max(abs(float(row.qrev_srmr_norm)), 1e-12) if np.isfinite(score) else np.nan,
                })
        except Exception as exc:
            srmr_bandwidth_errors.append({"logical_recording_id": recording_id, "error_type": type(exc).__name__, "message": str(exc)})

srmr_bandwidth = pd.DataFrame(srmr_bandwidth_rows)
srmr_bandwidth_error_table = pd.DataFrame(srmr_bandwidth_errors, columns=["logical_recording_id", "error_type", "message"])
save_table(srmr_bandwidth, VALIDATION / "qrev_v400_srmr_bandwidth_characterization")
save_table(srmr_bandwidth_error_table, AUDIT / "qrev_v400_srmr_bandwidth_errors", parquet=False)

g5_cohort_checks = pd.DataFrame([
    {"gate": "G5", "check": "SRMR bandwidth sensitivity characterized", "passed": bool(len(srmr_bandwidth) and set(pd.to_numeric(srmr_bandwidth["lowpass_cutoff_hz"], errors="coerce")) >= {3000.0, 4000.0, 6000.0, 7999.0}), "observed": len(srmr_bandwidth), "required": f">= {SRMR_BANDWIDTH_RECORDINGS * 4} rows"},
    {"gate": "G5", "check": "SRMR bandwidth characterization errors", "passed": srmr_bandwidth_error_table.empty, "observed": len(srmr_bandwidth_error_table), "required": 0},
])
save_table(g5_cohort_checks, VALIDATION / "qrev_v400_g5_cohort_checks", parquet=False)
display(g5_cohort_checks)
display(srmr_bandwidth.head())

In [ ]:
# G7/G8 — empirical behavior, censoring, persistence, participant weighting, and redundancy.
empirical_summary = empirical_feature_summary(recording_table) if len(recording_table) else pd.DataFrame()
save_table(empirical_summary, VALIDATION / "qrev_v400_empirical_feature_summary")

status_rows = []
for feature in ANALYSIS_FEATURES:
    status_column = f"{feature}_status"
    if status_column in recording_table:
        counts = recording_table[status_column].fillna("missing_status").astype(str).value_counts(dropna=False)
        for status, count in counts.items():
            status_rows.append({
                "feature": feature,
                "status": status,
                "recording_count": int(count),
                "fraction": float(count / max(1, len(recording_table))),
            })
status_summary = pd.DataFrame(status_rows)
save_table(status_summary, VALIDATION / "qrev_v400_status_missingness_summary")

censoring_summary = pd.DataFrame([
    {
        "level": "recording",
        "eligible_n": int(pd.to_numeric(recording_table.get("qrev_tail_persistence_median_sec_raw_estimate", pd.Series(dtype=float)), errors="coerce").notna().sum()),
        "right_censored_n": int(recording_table.get("qrev_persistence_recording_median_censored", pd.Series(dtype=bool)).fillna(False).astype(bool).sum()),
        "right_censored_fraction": float(recording_table.get("qrev_persistence_recording_median_censored", pd.Series(dtype=bool)).fillna(False).astype(bool).mean()) if len(recording_table) else np.nan,
        "horizon_sec": PARAMETERS.persistence_horizon_ms / 1000.0,
    },
    {
        "level": "boundary",
        "eligible_n": int(boundary_ledger.get("persistence_eligible", pd.Series(dtype=bool)).fillna(False).astype(bool).sum()),
        "right_censored_n": int(boundary_ledger.get("tail_persistence_right_censored", pd.Series(dtype=bool)).fillna(False).astype(bool).sum()),
        "right_censored_fraction": float(boundary_ledger.loc[boundary_ledger.get("persistence_eligible", pd.Series(index=boundary_ledger.index, dtype=bool)).fillna(False).astype(bool), "tail_persistence_right_censored"].astype(bool).mean()) if len(boundary_ledger) and "persistence_eligible" in boundary_ledger else np.nan,
        "horizon_sec": PARAMETERS.persistence_horizon_ms / 1000.0,
    },
])
save_table(censoring_summary, VALIDATION / "qrev_v400_censoring_summary", parquet=False)

repeat_persistence = repeated_recording_persistence(
    recording_table,
    subject_column=subject_column,
    date_column=date_column,
) if len(recording_table) else pd.DataFrame()
save_table(repeat_persistence, VALIDATION / "qrev_v400_repeated_recording_persistence")

participant_resampling = participant_balanced_resampling(
    recording_table,
    subject_column=subject_column,
    iterations=PARTICIPANT_BALANCED_ITERATIONS,
) if len(recording_table) else pd.DataFrame()
participant_summary = participant_balanced_summary(participant_resampling)
if participant_summary.empty:
    participant_summary = pd.DataFrame(columns=[
        "feature", "iterations", "median_of_medians", "p025_median", "p975_median",
        "median_availability_fraction", "p025_availability_fraction", "p975_availability_fraction",
    ])
save_table(participant_resampling, VALIDATION / "qrev_v400_participant_balanced_resampling")
save_table(participant_summary, VALIDATION / "qrev_v400_participant_balanced_summary")

recording_weighted_rows = []
for feature in ANALYSIS_FEATURES:
    values = pd.to_numeric(recording_table.get(feature, pd.Series(dtype=float)), errors="coerce").dropna()
    recording_weighted_rows.append({
        "feature": feature,
        "recording_weighted_median": float(values.median()) if len(values) else np.nan,
        "recording_weighted_availability": len(values) / max(1, len(recording_table)),
    })
recording_weighted = pd.DataFrame(recording_weighted_rows)
weighting_comparison = recording_weighted.merge(participant_summary, on="feature", how="left")
weighting_comparison["absolute_median_difference"] = (
    weighting_comparison["recording_weighted_median"] - weighting_comparison["median_of_medians"]
).abs()
save_table(weighting_comparison, VALIDATION / "qrev_v400_recording_vs_participant_weighting")

correlation_rows = []
if len(recording_table):
    for i, left in enumerate(ANALYSIS_FEATURES):
        for right in ANALYSIS_FEATURES[i + 1:]:
            pair = recording_table[[left, right]].apply(pd.to_numeric, errors="coerce").dropna()
            rho = float(stats.spearmanr(pair[left], pair[right]).statistic) if len(pair) >= 3 and pair[left].nunique() > 1 and pair[right].nunique() > 1 else np.nan
            correlation_rows.append({"feature_a": left, "feature_b": right, "pairwise_n": len(pair), "spearman_rho": rho, "absolute_spearman_rho": abs(rho) if np.isfinite(rho) else np.nan})
correlation_summary = pd.DataFrame(correlation_rows, columns=["feature_a", "feature_b", "pairwise_n", "spearman_rho", "absolute_spearman_rho"])
save_table(correlation_summary, VALIDATION / "qrev_v400_pairwise_redundancy")

range_checks = []
for feature in ANALYSIS_FEATURES:
    values = pd.to_numeric(recording_table.get(feature, pd.Series(dtype=float)), errors="coerce").dropna()
    if feature == "qrev_tail_persistence_median_sec":
        plausible = bool(((values >= 0) & (values <= PARAMETERS.persistence_horizon_ms / 1000.0 + 1e-12)).all())
    elif feature in ["qrev_downward_decay_rate_db_per_sec", "qrev_srmr_norm"]:
        plausible = bool((values > 0).all())
    else:
        plausible = bool(np.isfinite(values).all())
    range_checks.append({"gate": "G7", "check": f"plausible mathematical range: {feature}", "passed": plausible, "observed": f"n={len(values)} min={values.min() if len(values) else np.nan} max={values.max() if len(values) else np.nan}", "required": "feature-defined finite range"})

g7_checks = pd.DataFrame(range_checks + [
    {"gate": "G7", "check": "empirical summary complete", "passed": len(empirical_summary) == len(ANALYSIS_FEATURES), "observed": len(empirical_summary), "required": len(ANALYSIS_FEATURES)},
    {"gate": "G7", "check": "missingness reasons summarized", "passed": len(status_summary) > 0, "observed": len(status_summary), "required": ">0"},
    {"gate": "G7", "check": "censoring explicit at boundary and recording levels", "passed": set(censoring_summary["level"]) == {"boundary", "recording"}, "observed": censoring_summary["level"].tolist(), "required": ["boundary", "recording"]},
])
g8_checks = pd.DataFrame([
    {"gate": "G8", "check": "repeated-recording evidence complete", "passed": len(repeat_persistence) == len(ANALYSIS_FEATURES), "observed": len(repeat_persistence), "required": len(ANALYSIS_FEATURES)},
    {"gate": "G8", "check": "participant-balanced resampling complete", "passed": bool(len(participant_resampling) and participant_resampling["iteration"].nunique() == PARTICIPANT_BALANCED_ITERATIONS), "observed": participant_resampling.get("iteration", pd.Series(dtype=int)).nunique(), "required": PARTICIPANT_BALANCED_ITERATIONS},
    {"gate": "G8", "check": "all pairwise redundancy estimates include support", "passed": len(correlation_summary) == 6 and correlation_summary["pairwise_n"].notna().all(), "observed": len(correlation_summary), "required": 6},
])
save_table(g7_checks, VALIDATION / "qrev_v400_g7_checks", parquet=False)
save_table(g8_checks, VALIDATION / "qrev_v400_g8_checks", parquet=False)
display(empirical_summary)
display(censoring_summary)
display(repeat_persistence)
display(correlation_summary)

In [ ]:
# G10 preparation — support-aware, non-imputed ML interface and complete value/status contract.
ml_interface = model_interface_frame(recording_table) if len(recording_table) else pd.DataFrame()
save_table(ml_interface, TABLES / "qrev_v400_model_ready_features")

ml_check_rows = []
for feature in ANALYSIS_FEATURES:
    required = [
        feature,
        f"{feature}__available",
        f"{feature}__status",
        f"{feature}__missing_reason",
        f"{feature}__support_tier",
    ]
    ml_check_rows.append({
        "gate": "G10",
        "check": f"ML interface complete: {feature}",
        "passed": all(column in ml_interface for column in required),
        "observed": [column for column in required if column in ml_interface],
        "required": required,
    })
    if feature in ml_interface:
        available = ml_interface[f"{feature}__available"].astype(bool)
        missing_values = pd.to_numeric(ml_interface.loc[~available, feature], errors="coerce")
        ml_check_rows.append({
            "gate": "G10",
            "check": f"no imputation: {feature}",
            "passed": missing_values.isna().all(),
            "observed": int(missing_values.notna().sum()),
            "required": 0,
        })
ml_check_rows.extend([
    {"gate": "G10", "check": "persistence censoring exported", "passed": "qrev_persistence_recording_median_censored" in ml_interface, "observed": "qrev_persistence_recording_median_censored" in ml_interface, "required": True},
    {"gate": "G10", "check": "family scalar absent", "passed": bool("qrev_family_scalar_available" in ml_interface and not ml_interface["qrev_family_scalar_available"].astype(bool).any()), "observed": False, "required": False},
    {"gate": "G10", "check": "standalone rejection prohibited", "passed": bool("qrev_standalone_reject_allowed" in ml_interface and not ml_interface["qrev_standalone_reject_allowed"].astype(bool).any()), "observed": False, "required": False},
])
ml_checks = pd.DataFrame(ml_check_rows)
save_table(ml_checks, VALIDATION / "qrev_v400_ml_interface_checks", parquet=False)
display(ml_checks)

In [ ]:
# Panels D, E, F, H, and J — empirical publication figures with source data and provenance.
if RUN_COHORT_EXTRACTION and len(recording_table):
    # D1 — support-policy availability.
    d1 = support_policy_summary.copy()
    d1 = d1.loc[d1["feature"].isin(CONDITIONAL_BOUNDARY_FEATURES)].copy()
    fig, ax = plt.subplots(figsize=(9.5, 5.4))
    x = np.arange(len(CONDITIONAL_BOUNDARY_FEATURES))
    width = 0.23
    for offset, policy in enumerate(SUPPORT_POLICIES):
        local = d1.loc[d1["minimum_boundary_count"].eq(policy)].set_index("feature").reindex(CONDITIONAL_BOUNDARY_FEATURES)
        ax.bar(x + (offset - 1) * width, 100 * local["availability_fraction"].to_numpy(float), width=width, label=f"minimum {policy}")
    ax.set_xticks(x, ["Tail excess", "Persistence", "Decay rate"])
    ax.set_ylabel("Available recordings (%)")
    ax.set_title("QREV support-policy availability")
    ax.set_ylim(0, 100)
    ax.legend(frameon=False)
    figure_index_rows.append(save_figure_bundle(
        fig, stem="D1_support_policy_availability", panel="D", source_data=d1,
        scientific_question="How does conditional-feature availability change under two-, three-, and four-boundary support policies?",
        caption="Panel D1. Availability of the three boundary-conditioned QREV measurements under prespecified minimum support policies. Raw estimates remain retained even when analysis availability is suppressed. This panel informs, but does not make, the G10 support-policy decision.",
    ))
    plt.close(fig)

    # D2 — status and censoring regimes.
    d2 = status_summary.copy()
    fig, ax = plt.subplots(figsize=(10.5, 5.8))
    pivot = d2.pivot(index="feature", columns="status", values="fraction").fillna(0).reindex(ANALYSIS_FEATURES)
    bottom = np.zeros(len(pivot))
    for status in pivot.columns:
        values = 100 * pivot[status].to_numpy(float)
        ax.bar(np.arange(len(pivot)), values, bottom=bottom, label=status)
        bottom += values
    ax.set_xticks(np.arange(len(pivot)), ["Tail excess", "Persistence", "Decay", "SRMR"])
    ax.set_ylabel("Recordings (%)")
    ax.set_title("QREV measurement status, missingness, and censoring")
    ax.set_ylim(0, 100)
    ax.legend(frameon=False, fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
    figure_index_rows.append(save_figure_bundle(
        fig, stem="D2_status_missingness_censoring", panel="D", source_data=d2,
        scientific_question="What support, missingness, and right-censoring regimes occur in the corrected cohort?",
        caption="Panel D2. Recording-level QREV measurement statuses. Right-censored persistence is separated from exact measured persistence, and unavailable conditional values remain missing with reasons rather than being assigned zero.",
    ))
    plt.close(fig)

    # D3 — empirical precision versus valid pause support, plus SRMR speech support.
    precision_support_parts = []
    support_lookup = recording_table.set_index("logical_recording_id")
    for feature, spec in BOUNDARY_FEATURE_SPECS.items():
        local = bootstrap_precision.loc[bootstrap_precision["feature"].eq(feature)].copy()
        if len(local):
            local["valid_pause_support_sec"] = local["logical_recording_id"].map(
                pd.to_numeric(support_lookup[spec["support_sec"]], errors="coerce")
            )
            precision_support_parts.append(local)
    d3_precision = pd.concat(precision_support_parts, ignore_index=True) if precision_support_parts else pd.DataFrame()
    d3_srmr = recording_table[["logical_recording_id", "qrev_srmr_strict_speech_support_sec", "qrev_srmr_norm"]].copy()
    d3_srmr["feature"] = "qrev_srmr_norm"
    d3_srmr["available"] = pd.to_numeric(d3_srmr["qrev_srmr_norm"], errors="coerce").notna()
    d3_source = pd.concat([
        d3_precision.assign(source_type="boundary_bootstrap_precision"),
        d3_srmr.assign(source_type="srmr_speech_support"),
    ], ignore_index=True, sort=False)
    fig, axes = plt.subplots(2, 2, figsize=(11.2, 8.2))
    for ax, feature, label in zip(axes.flat[:3], CONDITIONAL_BOUNDARY_FEATURES, ["Tail excess", "Persistence", "Decay rate"]):
        local = d3_precision.loc[d3_precision["feature"].eq(feature)].copy()
        ax.scatter(local["valid_pause_support_sec"], local["bootstrap_ci95_width"], s=14, alpha=0.55)
        ax.set_xlabel("Valid pause support (s)")
        ax.set_ylabel("Bootstrap 95% CI width")
        ax.set_title(label)
    support_values = pd.to_numeric(d3_srmr["qrev_srmr_strict_speech_support_sec"], errors="coerce")
    axes.flat[3].hist([support_values[d3_srmr["available"]], support_values[~d3_srmr["available"]]], bins=20, stacked=True, label=["SRMR measured", "SRMR unavailable"])
    axes.flat[3].axvline(PARAMETERS.minimum_srmr_speech_support_sec, linestyle="--", linewidth=1.2, label="minimum support")
    axes.flat[3].set_xlabel("Strict-speech support (s)")
    axes.flat[3].set_ylabel("Recordings")
    axes.flat[3].set_title("SRMR support regime")
    axes.flat[3].legend(frameon=False, fontsize=8)
    figure_index_rows.append(save_figure_bundle(
        fig, stem="D3_support_precision_relationships", panel="D", source_data=d3_source,
        scientific_question="How do boundary-estimate precision and SRMR availability vary with independent support quantities?",
        caption="Panel D3. Boundary-level bootstrap precision versus valid pause duration for each conditional measurement, and pinned SRMR availability versus strict-speech support. Precision widths retain feature-specific units and are not combined into a family score.",
    ))
    plt.close(fig)

    # E1 — deletion robustness.
    e1 = delete_one_summary.copy()
    e1 = e1.loc[e1["minimum_boundary_count"].eq(2)].copy()
    fig, ax = plt.subplots(figsize=(9.5, 5.4))
    ax.bar(np.arange(len(e1)), e1["median_absolute_delta"].to_numpy(float))
    ax.errorbar(np.arange(len(e1)), e1["median_absolute_delta"], yerr=(e1["p95_absolute_delta"] - e1["median_absolute_delta"]).clip(lower=0), fmt="none", capsize=4)
    ax.set_xticks(np.arange(len(e1)), ["Tail excess", "Persistence", "Decay rate"])
    ax.set_ylabel("Absolute change after deleting one boundary")
    ax.set_title("QREV whole-boundary deletion sensitivity")
    figure_index_rows.append(save_figure_bundle(
        fig, stem="E1_delete_one_boundary_sensitivity", panel="E", source_data=e1,
        scientific_question="How strongly can a single eligible boundary influence each recording-level median?",
        caption="Panel E1. Median and 95th-percentile absolute change after omitting one eligible speech-to-pause boundary under the provisional two-boundary policy. Units remain feature-specific; bars are not directly comparable as a common severity scale.",
    ))
    plt.close(fig)

    # E2 — parameter and boundary sensitivity as availability agreement.
    e2 = robustness_summary.copy()
    e2 = e2.loc[~e2["variant"].eq("baseline")].copy()
    selected_variants = [
        "offset_minus_100ms", "offset_minus_50ms", "offset_plus_50ms", "offset_plus_100ms",
        "floor_600_900ms", "floor_800_1100ms", "horizon_400ms", "threshold_2db", "threshold_4db",
        "frame_20ms_hop10ms", "frame_40ms_hop10ms", "early_80ms", "early_120ms", "decay_200ms", "decay_400ms",
    ]
    e2 = e2.loc[e2["variant"].isin(selected_variants)].copy()
    matrix = e2.pivot(index="variant", columns="feature", values="availability_agreement_fraction").reindex(selected_variants)
    fig, ax = plt.subplots(figsize=(10.5, 8.2))
    image = ax.imshow(matrix.to_numpy(float), aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(np.arange(matrix.shape[1]), ["Tail excess", "Persistence", "Decay"])
    ax.set_yticks(np.arange(matrix.shape[0]), matrix.index)
    ax.set_title("QREV availability agreement under boundary and estimator perturbations")
    cbar = fig.colorbar(image, ax=ax)
    cbar.set_label("Availability agreement fraction")
    figure_index_rows.append(save_figure_bundle(
        fig, stem="E2_boundary_parameter_sensitivity", panel="E", source_data=e2,
        scientific_question="How stable are feature availability decisions under prespecified boundary, floor, horizon, frame, and window perturbations?",
        caption="Panel E2. Availability agreement relative to the frozen default estimator across prespecified perturbations. Detailed value deltas and paired support are provided in the source table; no universal tolerance is imposed before G10 review.",
    ))
    plt.close(fig)

    # E3 — paired value sensitivity normalized by each feature's empirical IQR.
    scale_map = {}
    for feature in CONDITIONAL_BOUNDARY_FEATURES:
        vals = pd.to_numeric(recording_table[feature], errors="coerce").dropna()
        scale = float(vals.quantile(0.75) - vals.quantile(0.25)) if len(vals) else np.nan
        scale_map[feature] = scale
    e3 = robustness_summary.loc[~robustness_summary["variant"].eq("baseline")].copy()
    e3["empirical_iqr"] = e3["feature"].map(scale_map)
    e3["median_absolute_delta_in_iqr_units"] = e3["median_absolute_delta"] / e3["empirical_iqr"].replace(0, np.nan)
    e3 = e3.loc[e3["variant"].isin(selected_variants)].copy()
    matrix_delta = e3.pivot(index="variant", columns="feature", values="median_absolute_delta_in_iqr_units").reindex(selected_variants)
    fig, ax = plt.subplots(figsize=(10.5, 8.2))
    finite_max = np.nanquantile(matrix_delta.to_numpy(float), 0.95) if np.isfinite(matrix_delta.to_numpy(float)).any() else 1.0
    image = ax.imshow(matrix_delta.to_numpy(float), aspect="auto", vmin=0, vmax=max(float(finite_max), 1e-6))
    ax.set_xticks(np.arange(matrix_delta.shape[1]), ["Tail excess", "Persistence", "Decay"] )
    ax.set_yticks(np.arange(matrix_delta.shape[0]), matrix_delta.index)
    ax.set_title("QREV paired value sensitivity normalized by empirical IQR")
    cbar = fig.colorbar(image, ax=ax)
    cbar.set_label("Median absolute change / empirical IQR")
    figure_index_rows.append(save_figure_bundle(
        fig, stem="E3_parameter_value_sensitivity", panel="E", source_data=e3,
        scientific_question="How large are paired estimator changes under prespecified perturbations relative to each feature's empirical spread?",
        caption="Panel E3. Median paired absolute change under each perturbation, divided by the corrected cohort IQR of the same feature. Normalization is for sensitivity visualization only and does not construct a QREV scalar or equate feature meanings.",
    ))
    plt.close(fig)

    # F — empirical distributions with availability, support, censoring, and outlier context.
    f_source_rows = []
    fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.2))
    display_names = ["Tail excess (dB)", "Persistence (s)", "Downward decay rate (dB/s)", "Normalized-fast SRMR"]
    for ax, feature, label in zip(axes.flat, ANALYSIS_FEATURES, display_names):
        values_all = pd.to_numeric(recording_table[feature], errors="coerce")
        values = values_all.dropna()
        status_col = f"{feature}_status"
        tier_col = f"{feature}_support_tier"
        for idx, row in recording_table.iterrows():
            f_source_rows.append({
                "logical_recording_id": row["logical_recording_id"],
                "feature": feature,
                "value": values_all.loc[idx],
                "available": bool(np.isfinite(values_all.loc[idx])),
                "status": str(row.get(status_col, "")),
                "support_tier": str(row.get(tier_col, "")),
                "right_censored": bool(row.get("qrev_persistence_recording_median_censored", False)) if feature == "qrev_tail_persistence_median_sec" else False,
            })
        ax.hist(values, bins=30)
        if len(values):
            ax.axvline(values.median(), linestyle="--", linewidth=1.5)
            p01, p99 = values.quantile([0.01, 0.99])
            ax.axvline(p01, linestyle=":", linewidth=0.9)
            ax.axvline(p99, linestyle=":", linewidth=0.9)
        ax.set_xlabel(label)
        ax.set_ylabel("Recordings")
        availability = len(values) / max(1, len(values_all))
        censored_n = int(recording_table.get("qrev_persistence_recording_median_censored", pd.Series(dtype=bool)).fillna(False).astype(bool).sum()) if feature == "qrev_tail_persistence_median_sec" else 0
        title = f"available={len(values)}/{len(values_all)} ({100*availability:.1f}%)"
        if len(values):
            title += f"; median={values.median():.3g}; p1-p99={p01:.3g}-{p99:.3g}"
        if censored_n:
            title += f"; censored={censored_n}"
        ax.set_title(title, fontsize=9)
    figure_index_rows.append(save_figure_bundle(
        fig, stem="F_empirical_distributions", panel="F", source_data=pd.DataFrame(f_source_rows),
        scientific_question="What ranges, masses, and availability regimes do corrected QREV measurements show in real recordings?",
        caption="Panel F. Recording-level distributions of the four QREV measurements. Panels use feature-specific axes and units. Persistence values at 0.6 s include explicit right-censored lower bounds and must not be interpreted as exact durations.",
    ))
    plt.close(fig)

    # H1 — repeated-recording persistence.
    h1 = repeat_persistence.copy()
    fig, ax = plt.subplots(figsize=(9.5, 5.5))
    ax.bar(np.arange(len(h1)) - 0.16, h1["first_second_spearman"].to_numpy(float), width=0.32, label="Spearman")
    ax.bar(np.arange(len(h1)) + 0.16, h1["icc1_first_two"].to_numpy(float), width=0.32, label="ICC(1)")
    ax.axhline(0, linewidth=0.8)
    ax.set_xticks(np.arange(len(h1)), ["Tail excess", "Persistence", "Decay", "SRMR"])
    ax.set_ylabel("Reliability coefficient")
    ax.set_title("QREV repeated-recording persistence")
    ax.legend(frameon=False)
    figure_index_rows.append(save_figure_bundle(
        fig, stem="H1_repeated_recording_persistence", panel="H", source_data=h1,
        scientific_question="How persistent are QREV measurements across the first two recordings per participant?",
        caption="Panel H1. Participant-aware first-versus-second recording Spearman correlations and ICC(1). Pairwise sample sizes, absolute differences, and uncensored persistence analyses are retained in the source table.",
    ))
    plt.close(fig)

    # H2 — redundancy with pairwise support.
    h2 = correlation_summary.copy()
    fig, ax = plt.subplots(figsize=(9.5, 5.8))
    labels = [f"{a.replace('qrev_','')} vs\n{b.replace('qrev_','')}" for a, b in zip(h2["feature_a"], h2["feature_b"])]
    bars = ax.bar(np.arange(len(h2)), h2["spearman_rho"].to_numpy(float))
    ax.axhline(0, linewidth=0.8)
    ax.set_xticks(np.arange(len(h2)), labels, rotation=25, ha="right")
    ax.set_ylabel("Spearman rho")
    ax.set_title("QREV within-family dependence with pairwise support")
    for bar, n in zip(bars, h2["pairwise_n"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"n={int(n)}", ha="center", va="bottom" if bar.get_height() >= 0 else "top", fontsize=8)
    figure_index_rows.append(save_figure_bundle(
        fig, stem="H2_redundancy_and_convergent_evidence", panel="H", source_data=h2,
        scientific_question="Are boundary proxies redundant, and how does pinned SRMR relate to them under pairwise availability?",
        caption="Panel H2. Pairwise Spearman dependence among QREV measurements with pairwise available sample sizes. Structural agreement is not required, and correlated boundary features are not treated as independent causal evidence.",
    ))
    plt.close(fig)

    # H3 — participant versus recording weighting.
    h3 = weighting_comparison.copy()
    fig, ax = plt.subplots(figsize=(9.5, 5.4))
    ax.bar(np.arange(len(h3)), h3["absolute_median_difference"].to_numpy(float))
    ax.set_xticks(np.arange(len(h3)), ["Tail excess", "Persistence", "Decay", "SRMR"])
    ax.set_ylabel("Absolute median difference (feature units)")
    ax.set_title("Recording-weighted versus participant-balanced summaries")
    figure_index_rows.append(save_figure_bundle(
        fig, stem="H3_participant_weighting", panel="H", source_data=h3,
        scientific_question="Do participants with more recordings materially shift cohort summaries?",
        caption="Panel H3. Absolute difference between recording-weighted and one-recording-per-participant resampled medians. Values remain in feature-specific units and are interpreted per feature rather than as a shared scale.",
    ))
    plt.close(fig)

    # J — ML interface completeness.
    j_rows = []
    for feature in ANALYSIS_FEATURES:
        j_rows.append({
            "feature": feature,
            "value_present": feature in ml_interface,
            "availability_present": f"{feature}__available" in ml_interface,
            "status_present": f"{feature}__status" in ml_interface,
            "missing_reason_present": f"{feature}__missing_reason" in ml_interface,
            "support_tier_present": f"{feature}__support_tier" in ml_interface,
            "available_fraction": float(ml_interface[f"{feature}__available"].astype(bool).mean()),
        })
    j_source = pd.DataFrame(j_rows)
    fields = ["value_present", "availability_present", "status_present", "missing_reason_present", "support_tier_present"]
    fig, ax = plt.subplots(figsize=(9.5, 4.8))
    image = ax.imshow(j_source[fields].astype(int).to_numpy(), aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(np.arange(len(fields)), ["Value", "Available", "Status", "Missing reason", "Support tier"])
    ax.set_yticks(np.arange(len(j_source)), ["Tail excess", "Persistence", "Decay", "SRMR"])
    ax.set_title("QREV quality-aware ML interface contract")
    cbar = fig.colorbar(image, ax=ax, ticks=[0,1])
    cbar.ax.set_yticklabels(["Absent", "Present"])
    figure_index_rows.append(save_figure_bundle(
        fig, stem="J_ml_handoff_contract", panel="J", source_data=j_source,
        scientific_question="Does the ML handoff preserve values, availability, reasons, support, and censoring without imputation or a family scalar?",
        caption="Panel J. Completeness of the QREV quality-aware ML interface. The export preserves non-imputed values, availability, status, missing reason, support tier, persistence censoring, and version metadata; it does not construct a generic quality score or operational threshold.",
    ))
    plt.close(fig)

figure_index = pd.DataFrame(figure_index_rows)
save_table(figure_index, TABLES / "qrev_v400_figure_index", parquet=False)
display(figure_index)

In [ ]:
# Panel G — deterministic, signal-derived real-recording examples with no causal source labels.
gallery_rows = []
gallery_errors = []
if RUN_COHORT_EXTRACTION and BUILD_GALLERY and len(recording_table):
    candidates = []
    working = recording_table.copy()

    def add_candidate(reason, frame, sort_column, ascending, condition=None):
        local = frame.copy()
        if condition is not None:
            local = local.loc[condition(local)]
        local[sort_column] = pd.to_numeric(local[sort_column], errors="coerce")
        local = local.dropna(subset=[sort_column]).sort_values(
            [sort_column, "logical_recording_id"],
            ascending=[ascending, True],
        )
        if len(local):
            candidates.append({
                "logical_recording_id": str(local.iloc[0]["logical_recording_id"]),
                "selection_reason": reason,
            })

    add_candidate("low_available_tail_excess", working, "qrev_tail_excess_100ms_db", True)
    add_candidate("high_available_tail_excess", working, "qrev_tail_excess_100ms_db", False)
    add_candidate(
        "short_uncensored_persistence", working, "qrev_tail_persistence_median_sec", True,
        lambda x: ~x["qrev_persistence_recording_median_censored"].fillna(False).astype(bool),
    )
    add_candidate(
        "long_uncensored_persistence", working, "qrev_tail_persistence_median_sec", False,
        lambda x: ~x["qrev_persistence_recording_median_censored"].fillna(False).astype(bool),
    )
    add_candidate(
        "right_censored_persistence", working, "qrev_persistence_right_censored_fraction", False,
        lambda x: x["qrev_persistence_recording_median_censored"].fillna(False).astype(bool),
    )
    add_candidate("slow_valid_downward_decay", working, "qrev_downward_decay_rate_db_per_sec", True)
    add_candidate("fast_valid_downward_decay", working, "qrev_downward_decay_rate_db_per_sec", False)
    add_candidate("low_srmr", working, "qrev_srmr_norm", True)
    add_candidate("high_srmr", working, "qrev_srmr_norm", False)
    add_candidate(
        "conditional_features_unavailable", working, "qrev_internal_boundary_count", True,
        lambda x: x["qrev_family_status"].astype(str).eq("primary_unavailable"),
    )

    selection = (
        pd.DataFrame(candidates)
        .drop_duplicates("logical_recording_id")
        .head(GALLERY_RECORDING_LIMIT)
    )
    save_table(selection, GALLERIES / "qrev_v400_gallery_selection", parquet=False)
    frozen_lookup = frozen.set_index("logical_recording_id")
    ledger_groups = {
        str(key): group
        for key, group in boundary_ledger.groupby("logical_recording_id", sort=False)
    }

    for selected in selection.itertuples(index=False):
        recording_id = str(selected.logical_recording_id)
        try:
            source_row = frozen_lookup.loc[recording_id]
            media_path = resolve_media_path(
                source_row["media_path"],
                media_root_override=Path(MEDIA_ROOT_OVERRIDE) if MEDIA_ROOT_OVERRIDE else None,
                media_path_map=MEDIA_PATH_MAP,
            )
            views = decode_audio_views(
                media_path, ffmpeg=ffmpeg, ffprobe=ffprobe, analysis_rate=FS
            )
            waveform = analysis_waveform_from_audio_views(views)
            local_ledger = ledger_groups.get(recording_id, pd.DataFrame()).copy()

            if len(local_ledger):
                eligible = local_ledger.loc[
                    local_ledger["tail_eligible"].fillna(False).astype(bool)
                ].copy()
                boundary_row = (
                    eligible.sort_values(["tail_excess_100ms_db", "boundary_id"]).iloc[len(eligible) // 2]
                    if len(eligible)
                    else local_ledger.sort_values("boundary_id").iloc[0]
                )
                offset = float(boundary_row["speech_offset_sec"])
                pause_end = float(boundary_row["pause_end_sec"])
                boundary_id = str(boundary_row["boundary_id"])
                trace = boundary_envelope_trace(
                    waveform, FS, offset, pause_end, parameters=PARAMETERS
                )
                plot_start = max(0.0, offset - 0.25)
                plot_end = min(len(waveform) / FS, offset + 1.05)
            else:
                offset = min(max(len(waveform) / FS / 2.0, 0.25), max(len(waveform) / FS - 0.25, 0.25))
                pause_end = min(len(waveform) / FS, offset + 1.0)
                boundary_id = "N/A_no_internal_boundary"
                trace = pd.DataFrame(columns=["relative_mid_sec", "ac_rms_dbfs"])
                plot_start = max(0.0, offset - 0.65)
                plot_end = min(len(waveform) / FS, offset + 0.65)

            left = int(np.floor(plot_start * FS))
            right = int(np.ceil(plot_end * FS))
            audio = waveform[left:right]
            relative_times = np.arange(left, right, dtype=float) / FS - offset

            # Source-data waveform is deterministically thinned only for CSV size;
            # the plotted trace uses the same thinned coordinates.
            waveform_stride = max(1, int(np.ceil(max(len(audio), 1) / 5000)))
            wave_source = pd.DataFrame({
                "row_type": "waveform",
                "logical_recording_id": recording_id,
                "selection_reason": selected.selection_reason,
                "boundary_id": boundary_id,
                "relative_time_sec": relative_times[::waveform_stride],
                "frequency_hz": np.nan,
                "amplitude": audio[::waveform_stride],
                "ac_rms_dbfs": np.nan,
                "power_db": np.nan,
            })

            nperseg = int(min(512, max(64, len(audio))))
            noverlap = int(min(nperseg - 1, round(0.75 * nperseg)))
            frequencies, spec_times, spectrum = signal.spectrogram(
                audio,
                fs=FS,
                window="hann",
                nperseg=nperseg,
                noverlap=noverlap,
                detrend="constant",
                scaling="spectrum",
                mode="psd",
            )
            power_db = 10.0 * np.log10(np.maximum(spectrum, np.finfo(float).tiny))
            spec_relative_times = spec_times + plot_start - offset
            spec_source = pd.DataFrame({
                "row_type": "spectrogram",
                "logical_recording_id": recording_id,
                "selection_reason": selected.selection_reason,
                "boundary_id": boundary_id,
                "relative_time_sec": np.tile(spec_relative_times, len(frequencies)),
                "frequency_hz": np.repeat(frequencies, len(spec_relative_times)),
                "amplitude": np.nan,
                "ac_rms_dbfs": np.nan,
                "power_db": power_db.reshape(-1),
            })

            if len(trace):
                envelope_source = pd.DataFrame({
                    "row_type": "envelope",
                    "logical_recording_id": recording_id,
                    "selection_reason": selected.selection_reason,
                    "boundary_id": boundary_id,
                    "relative_time_sec": pd.to_numeric(trace["relative_mid_sec"], errors="coerce"),
                    "frequency_hz": np.nan,
                    "amplitude": np.nan,
                    "ac_rms_dbfs": pd.to_numeric(trace["ac_rms_dbfs"], errors="coerce"),
                    "power_db": np.nan,
                })
            else:
                envelope_source = pd.DataFrame(columns=wave_source.columns)

            source = pd.concat(
                [wave_source, spec_source, envelope_source],
                ignore_index=True,
                sort=False,
            )
            source["waveform_plot_start_sec"] = plot_start
            source["waveform_plot_end_sec"] = plot_end
            source["primary_speech_offset_sec"] = offset
            source["pause_end_sec"] = pause_end
            source["source_media_sha256"] = sha256_file(media_path)

            fig, axes = plt.subplots(
                3, 1, figsize=(10.8, 8.3), sharex=True,
                gridspec_kw={"height_ratios": [1.0, 1.35, 1.15]},
            )
            axes[0].plot(
                wave_source["relative_time_sec"],
                wave_source["amplitude"],
                linewidth=0.7,
            )
            axes[0].axvline(0, linestyle="--", linewidth=1.2, label="primary-speech offset")
            axes[0].set_ylabel("Amplitude")
            axes[0].set_title(f"{recording_id} — {selected.selection_reason}")
            axes[0].legend(frameon=False, fontsize=8)

            mesh = axes[1].pcolormesh(
                spec_relative_times,
                frequencies,
                power_db,
                shading="auto",
            )
            axes[1].axvline(0, linestyle="--", linewidth=1.0)
            axes[1].set_ylim(0, min(8000, FS / 2))
            axes[1].set_ylabel("Frequency (Hz)")
            fig.colorbar(mesh, ax=axes[1], pad=0.01, label="Power (dB)")

            if len(trace):
                axes[2].plot(
                    trace["relative_mid_sec"],
                    trace["ac_rms_dbfs"],
                    marker=".",
                    linewidth=1.0,
                )
                floor_value = pd.to_numeric(
                    pd.Series([boundary_row.get("floor_dbfs", np.nan)]),
                    errors="coerce",
                ).iloc[0]
                if np.isfinite(floor_value):
                    axes[2].axhline(
                        float(floor_value), linestyle="--", linewidth=1.0,
                        label="late-pause floor",
                    )
                axes[2].axvspan(
                    0.0,
                    PARAMETERS.early_tail_end_ms / 1000.0,
                    alpha=0.15,
                    label="early tail",
                )
                axes[2].axvspan(
                    PARAMETERS.floor_start_ms / 1000.0,
                    PARAMETERS.floor_end_ms / 1000.0,
                    alpha=0.15,
                    label="independent floor",
                )
                axes[2].axvline(
                    PARAMETERS.persistence_horizon_ms / 1000.0,
                    linestyle=":",
                    linewidth=1.0,
                    label="persistence horizon",
                )
                axes[2].legend(frameon=False, fontsize=8, ncol=2)
            else:
                axes[2].text(
                    0.5, 0.5,
                    "No internal boundary trace available",
                    transform=axes[2].transAxes,
                    ha="center", va="center",
                )
            axes[2].set_xlabel("Time from selected primary-speech offset (s)")
            axes[2].set_ylabel("AC RMS (dBFS)")

            safe_reason = str(selected.selection_reason).replace(" ", "_")
            stem = f"G_{safe_reason}_{recording_id}"
            png = GALLERIES / f"{stem}.png"
            svg = GALLERIES / f"{stem}.svg"
            pdf = GALLERIES / f"{stem}.pdf"
            csv = GALLERIES / f"{stem}.source.csv"
            cap = GALLERIES / f"{stem}.caption.md"
            prov = GALLERIES / f"{stem}.provenance.json"
            fig.tight_layout()
            fig.savefig(png, dpi=300, bbox_inches="tight")
            fig.savefig(svg, bbox_inches="tight")
            fig.savefig(pdf, bbox_inches="tight")
            source.to_csv(csv, index=False)
            cap.write_text(
                f"Panel G example. Deterministically selected recording `{recording_id}` for the signal-derived stratum `{selected.selection_reason}`. The three aligned views show waveform, spectrogram, and AC-RMS residual envelope around a natural primary-speech offset. Selection is label-blind and descriptive; no reverberation, breath, noise, or echo source identity is assigned.\n",
                encoding="utf-8",
            )
            write_json({
                "panel": "G",
                "figure_id": stem,
                "logical_recording_id": recording_id,
                "selection_reason": selected.selection_reason,
                "selection_is_label_blind": True,
                "causal_source_label_assigned": False,
                "views": ["waveform", "spectrogram", "ac_rms_envelope"],
                "spectrogram": {
                    "method": "scipy.signal.spectrogram",
                    "window": "hann",
                    "nperseg": nperseg,
                    "noverlap": noverlap,
                    "detrend": "constant",
                    "scaling": "spectrum",
                    "mode": "psd",
                },
                "measurement_version": MEASUREMENT_VERSION,
                "source_media_relative_path": str(source_row["media_path"]),
                "source_media_sha256": sha256_file(media_path),
                "source_csv": csv.name,
                "source_csv_sha256": sha256_file(csv),
                "created_utc": datetime.now(timezone.utc).isoformat(),
            }, prov)
            plt.close(fig)
            gallery_rows.append({
                "panel": "G",
                "figure_id": stem,
                "logical_recording_id": recording_id,
                "selection_reason": selected.selection_reason,
                "views": "waveform;spectrogram;ac_rms_envelope",
                "png": png.name,
                "svg": svg.name,
                "pdf": pdf.name,
                "source": csv.name,
                "caption": cap.name,
                "provenance": prov.name,
            })
        except Exception as exc:
            gallery_errors.append({
                "logical_recording_id": recording_id,
                "selection_reason": selected.selection_reason,
                "error_type": type(exc).__name__,
                "message": str(exc),
            })

gallery_index = pd.DataFrame(gallery_rows)
gallery_error_table = pd.DataFrame(
    gallery_errors,
    columns=["logical_recording_id", "selection_reason", "error_type", "message"],
)
save_table(gallery_index, GALLERIES / "qrev_v400_gallery_index", parquet=False)
save_table(gallery_error_table, AUDIT / "qrev_v400_gallery_errors", parquet=False)
if len(gallery_index):
    for row in gallery_index.to_dict("records"):
        figure_index_rows.append({**row, "stage": "COHORT", "status": "COMPLETE"})
    figure_index = pd.DataFrame(figure_index_rows)
    save_table(figure_index, TABLES / "qrev_v400_figure_index", parquet=False)
display(gallery_index)
display(gallery_error_table)

In [ ]:
# Complete the checklist evidence package and candidate manifest. G10 remains pending.
feature_decisions = pd.DataFrame([
    {
        "feature": "qrev_tail_excess_100ms_db",
        "provisional_role": "primary conditional residual-magnitude candidate",
        "g10_decision": "PENDING_COHORT_REVIEW",
        "claim": "signed early 0-100-ms residual level relative to an independent 0.7-1.0-s late-pause floor",
        "standalone_gate_allowed": False,
    },
    {
        "feature": "qrev_tail_persistence_median_sec",
        "provisional_role": "primary conditional bounded/censored persistence candidate",
        "g10_decision": "PENDING_COHORT_REVIEW",
        "claim": "time to sustained within-floor return, right-censored at 0.6 s; not reverberation time",
        "standalone_gate_allowed": False,
    },
    {
        "feature": "qrev_downward_decay_rate_db_per_sec",
        "provisional_role": "secondary conditional decay-shape candidate",
        "g10_decision": "PENDING_COHORT_REVIEW",
        "claim": "magnitude of a valid negative robust 0-300-ms envelope slope; unavailable is not zero",
        "standalone_gate_allowed": False,
    },
    {
        "feature": "qrev_srmr_norm",
        "provisional_role": "secondary published no-reference comparator",
        "g10_decision": "PENDING_COHORT_REVIEW",
        "claim": "pinned normalized-fast SRMR; reverberation-sensitive but not reverberation-specific",
        "standalone_gate_allowed": False,
    },
])
save_table(feature_decisions, VALIDATION / "qrev_v400_g10_feature_decisions")

figure_index = pd.DataFrame(figure_index_rows)
save_table(figure_index, TABLES / "qrev_v400_figure_index", parquet=False)
completed_panels = set(figure_index.get("panel", pd.Series(dtype=str)).astype(str))
gallery_complete = bool(not BUILD_GALLERY or (len(gallery_index) >= 8 and gallery_error_table.empty))
required_panels = {"A", "B", "C", "D", "E", "F", "G", "H", "J"}

figure_contract = pd.DataFrame([
    {"panel": panel, "status": "COMPLETE" if panel in completed_panels else "MISSING", "purpose": purpose}
    for panel, purpose in [
        ("A", "controlled construct response"), ("B", "discriminant specificity"),
        ("C", "transformation contract"), ("D", "support, availability, and censoring"),
        ("E", "boundary, horizon, floor, frame, and window sensitivity"),
        ("F", "empirical distributions"), ("G", "deterministic signal-linked examples"),
        ("H", "reliability, redundancy, and participant weighting"),
        ("J", "quality-aware ML handoff"),
    ]
] + [{"panel": "I", "status": "N/A", "purpose": "no retained discrete event detector"}])
save_table(figure_contract, VALIDATION / "qrev_v400_figure_contract", parquet=False)

all_cohort_check_frames = [
    input_checks, extraction_checks, g6_foundational_checks, g6_sensitivity_checks,
    g5_cohort_checks, g7_checks, g8_checks, ml_checks,
]
cohort_checks = pd.concat(all_cohort_check_frames, ignore_index=True, sort=False)
save_table(cohort_checks, VALIDATION / "qrev_v400_cohort_checks", parquet=False)

cohort_complete = bool(
    RUN_COHORT_EXTRACTION
    and len(recording_table) == len(frozen)
    and extraction_errors.empty
    and media_audit.get("sha256_match", pd.Series(dtype=bool)).astype(bool).all()
)
evidence_complete = bool(
    cohort_complete
    and cohort_checks["passed"].fillna(False).astype(bool).all()
    and required_panels.issubset(completed_panels)
    and gallery_complete
)

gate_summary = preflight_gates.copy()
gate_summary.loc[gate_summary["gate"].eq("G1"), "status"] = "PASS" if input_checks["passed"].all() else "FAIL"
gate_summary.loc[gate_summary["gate"].eq("G2"), "status"] = "PASS" if reconstruction_pass and extraction_checks.loc[extraction_checks["gate"].eq("G2"), "passed"].all() else "FAIL"
gate_summary.loc[gate_summary["gate"].eq("G5"), "status"] = "CONDITIONAL_EVIDENCE_COMPLETE" if g5_cohort_checks["passed"].all() else "FAIL"
gate_summary.loc[gate_summary["gate"].eq("G6"), "status"] = "EVIDENCE_COMPLETE_PENDING_REVIEW" if g6_foundational_checks["passed"].all() and g6_sensitivity_checks["passed"].all() else "FAIL"
gate_summary.loc[gate_summary["gate"].eq("G7"), "status"] = "EVIDENCE_COMPLETE_PENDING_REVIEW" if extraction_checks.loc[extraction_checks["gate"].eq("G7"), "passed"].all() and g7_checks["passed"].all() and gallery_complete else "FAIL"
gate_summary.loc[gate_summary["gate"].eq("G8"), "status"] = "EVIDENCE_COMPLETE_PENDING_REVIEW" if g8_checks["passed"].all() else "FAIL"
gate_summary.loc[gate_summary["gate"].eq("G9"), "status"] = "N/A"
gate_summary.loc[gate_summary["gate"].eq("G10"), "status"] = "PENDING_SCIENTIFIC_REVIEW"
save_table(gate_summary, VALIDATION / "qrev_v400_gate_summary_cohort", parquet=False)

candidate_manifest = {
    "measurement_version": MEASUREMENT_VERSION,
    "family": "QREV",
    "family_display_name": "Reverberation / residual-tail manifestations",
    "orchestration_version": COHORT_ORCHESTRATION_VERSION,
    "candidate_only": True,
    "cohort_extraction_completed": cohort_complete,
    "cohort_evidence_complete": evidence_complete,
    "recording_count": int(len(recording_table)),
    "participant_count": int(recording_table[subject_column].nunique(dropna=True)) if len(recording_table) else 0,
    "preflight_blocking_checks_pass": bool(preflight_manifest.get("preflight_blocking_checks_pass", False)),
    "preflight_hotfix_revision": preflight_manifest.get("preflight_hotfix_revision"),
    "global_dc_removal_enforced": bool(preflight_manifest.get("global_dc_removal_enforced", False)),
    "srmr_runtime_available": bool(preflight_manifest.get("srmr_runtime_available", False)),
    "package_tests_passed": bool(package_tests_passed),
    "srmr_variant": SRMR_VARIANT,
    "srmr_upstream_commit": SRMR_UPSTREAM_COMMIT,
    "gammatone_version": SRMR_GAMMATONE_VERSION,
    "scientific_review_decision": SCIENTIFIC_REVIEW_DECISION,
    "scientific_reviewer": SCIENTIFIC_REVIEWER,
    "scientific_review_rationale": SCIENTIFIC_REVIEW_RATIONALE,
    "freeze_allowed": False,
    "freeze_blocker": "feature-specific G10 scientific review after cohort audit",
    "analysis_features": list(ANALYSIS_FEATURES),
    "support_policy_status": "2/3/4 evidence complete; final policy pending G10",
    "boundary_contract": {
        "profile": CANONICAL_PROFILE,
        "natural_offset_view": CANONICAL_PRIMARY_VIEW,
        "srmr_support_view": CANONICAL_STRICT_VIEW,
        "view_fallback_allowed": False,
        "duplicate_guard_applied": False,
    },
    "persistence_contract": {
        "horizon_sec": PARAMETERS.persistence_horizon_ms / 1000.0,
        "floor_window_sec": [PARAMETERS.floor_start_ms/1000.0, PARAMETERS.floor_end_ms/1000.0],
        "right_censoring_explicit": True,
    },
    "required_panels_complete": required_panels.issubset(completed_panels) and gallery_complete,
    "completed_panels": sorted(completed_panels),
    "panel_i_status": "N/A_no_retained_event_detector",
    "figure_count": int(len(figure_index)),
    "family_scalar_constructed": False,
    "standalone_gate_allowed": False,
    "decision_threshold_status": "not_calibrated",
    "feature_values_recomputed_by_figures": False,
    "input_artifact_sha256": {row.artifact: row.sha256 for row in input_artifacts.itertuples(index=False)},
    "implementation_sha256": sha256_file(REVIEWED_SRC / "paper1_qc_reviewed" / "qrev_v400.py"),
    "cohort_orchestration_sha256": sha256_file(REVIEWED_SRC / "paper1_qc_reviewed" / "qrev_v400_cohort.py"),
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
write_json(candidate_manifest, MANIFESTS / "qrev_v400_cohort_candidate_manifest.json")
save_table(hash_inventory(STAGE), MANIFESTS / "qrev_v400_candidate_artifact_inventory", parquet=False)

display(gate_summary)
display(figure_contract)
display(feature_decisions)
print("QREV v4.0.0 REVIEWED COHORT RUN COMPLETE")
print(json.dumps(json_safe(candidate_manifest), indent=2))

if PUBLISH_AND_FREEZE:
    raise RuntimeError(
        "This cohort notebook cannot freeze QREV. Run the post-audit finalization/freeze patch only after G10 decisions."
    )

## Required next action

Save this fully executed notebook, close JupyterLab, and package the notebook plus the complete `outputs/reviewed/reverberation/qrev-v4.0.0-candidate` directory for independent post-cohort scientific review. The next review completes the ten-domain dashboard, the G1–G10 checklist, final feature roles, and freeze authorization. Do not publish or freeze from this notebook.